### Data Preparation for Wind Offshore ML-Model

In this file, I carry out the full data preparation cycle for machine learning aimed at forecasting offshore wind farm output. It takes raw weather forecasts, installed capacity data, and actual generation figures; cleans them; creates advanced physical metrics; and consolidates everything into a single dataset with a 15-minute time step.

Since the **Open-Meteo Historical Forecast API** has request limits, it was necessary to split the dataset into chunks.

In [1]:
# Imports
import pandas as pd
import os
import pyrootutils
from dotenv import load_dotenv

# Needed to load variables from the .env file into the environment
load_dotenv()

# Determining the project root based on the .gitignore file
PROJECT_ROOT = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)
from utils.offshore_utils import fetch_offshore_weather

# Determining data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean_data"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"

# Determining the time interval for offshore weather data collection
START = '2026-01-01'
END = '2026-06-20'

# # Uncomment if necessary============================================================
# # Fetching offshore weather data
# df = fetch_offshore_weather(start_date=START, end_date=END)

# # Forming the file name and saving the data in parquet format
# raw_weather_file_name =  f'offshore_ECMWF_singleruns_only_{START}_to_{END}.parquet'
# df.to_parquet(RAW_DATA_DIR / raw_weather_file_name)

Duplicate detection and display configuration. Since I included a one-hour overlap on either side of the 24-hour period, it is necessary to check for duplicates based on the `datetime_run_utc` and `datetime_utc` keys for each meteorological probe (`point_id`)

In [2]:
# The dataset was split by year due to export limitations of the Open-Meteo API
df_2024 = pd.read_parquet(RAW_DATA_DIR / 'offshore_ECMWF_singleruns_only_2024-03-01_to_2025-01-01.parquet')
df_2025 = pd.read_parquet(RAW_DATA_DIR / 'offshore_ECMWF_singleruns_only_2025-01-01_to_2026-01-01.parquet')
df_2026 = pd.read_parquet(RAW_DATA_DIR / 'offshore_ECMWF_singleruns_only_2026-01-01_to_2026-06-20.parquet')

# Duplicate search
dup_count_1 = df_2024.duplicated(subset=["datetime_run_utc", "datetime_utc", "point_id"]).sum()
dup_count_2 = df_2025.duplicated(subset=["datetime_run_utc", "datetime_utc", "point_id"]).sum()
dup_count_3 = df_2026.duplicated(subset=["datetime_run_utc", "datetime_utc", "point_id"]).sum()

# Displaying the number of duplicates in each dataset
print(f"Duplicates in df1: {dup_count_1}")
print(f"Duplicates in df2: {dup_count_2}")
print(f"Duplicates in df3: {dup_count_3}")


# Setting display options for pandas DataFrames
pd.set_option('display.max_columns', None)      # Show all columns in the DataFrame
pd.set_option('display.max_colwidth', None)     # Show full content of each column without truncation
pd.set_option('display.width', None)            # Automatically configure screen width

Duplicates in df1: 0
Duplicates in df2: 0
Duplicates in df3: 0


In [3]:
# Concatenating dataframes into one large dataframe to create a dataset
df_complete = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)
df_complete = df_complete.sort_values(by=["point_id", "datetime_run_utc", "datetime_utc"]).reset_index(drop=True)

# Another check for duplicates
dup_mask_complete = df_complete.duplicated(subset=["datetime_run_utc", "datetime_utc", "point_id"])
print(f"Duplicates in df_complete before cleaning: {dup_mask_complete.sum()}\n\n")

# Cleaning
df_complete = df_complete.drop_duplicates(
    subset=["datetime_run_utc", "datetime_utc", "point_id"], 
    keep='last'
)
dup_mask_complete = df_complete.duplicated(subset=["datetime_run_utc", "datetime_utc", "point_id"])
print(f"Duplicates in df_complete after cleaning: {dup_mask_complete.sum()}\n\n")

# Checking for missing values in the final dataframe
print(f'Number of missing values per column:')
print(df_complete.isna().sum())

# Checking the length of the final dataframe
# 302106 1H rows for each probe, this is ~900 days of data
print(f'Length of df_complete: {df_complete.shape[0]}')

# # Uncomment if necessary============================================================
# # Displaying a section of the table to verify how the 1-hour overlap works.
# filter_start_date   = pd.to_datetime('2025-12-31', utc=True)
# filter_end_date     = pd.to_datetime('2026-01-02', utc=True)
# df_complete[(df_complete['datetime_utc'] > filter_start_date) & (df_complete['datetime_utc'] < filter_end_date)].head(60)

# # Some rows are missing in the final dataframe, this is due to the fact that the data collection function does not add rows
# # where there are NaN values, this is not a problem, but it should be remembered when preparing the training dataset, and especially
# # when working with temporal lags

Duplicates in df_complete before cleaning: 728


Duplicates in df_complete after cleaning: 0


Number of missing values per column:
datetime_run_utc    0
datetime_utc        0
datetime_berlin     0
point_id            0
region              0
latitude            0
longitude           0
wind_speed          0
wind_direction      0
height_m            0
temperature_2m      0
temperature_hub     0
surface_pressure    0
air_density         0
precipitation       0
dtype: int64
Length of df_complete: 301378


In [5]:
# Key columns that define the uniqueness of an observation
key_cols = ['datetime_run_utc', 'datetime_utc', 'point_id']

# Count the number of rows that are duplicated based on the key columns
key_dups = df_complete.duplicated(subset=key_cols, keep=False).sum()
full_dups = df_complete.duplicated(keep=False).sum()

print(f"Rows with matching keys (time + point): {key_dups}")
print(f"Rows with identical values in all columns: {full_dups}")

if key_dups == full_dups:
    print("Success: All duplicates are exact copies. Pivot and GroupBy did not distort the data.")
else:
    print(f"Warning: Found {key_dups - full_dups} rows with identical keys but different values in the features!")

Rows with matching keys (time + point): 0
Rows with identical values in all columns: 0
Success: All duplicates are exact copies. Pivot and GroupBy did not distort the data.


In [ ]:
# Displaying the table structure
df_complete.sample(10)

,datetime_run_utc,datetime_utc,datetime_berlin,point_id,region,latitude,longitude,wind_speed,wind_direction,height_m,temperature_2m,temperature_hub,surface_pressure,air_density,precipitation
135676,2024-11-07 00:00:00+00:00,2024-11-08 12:00:00+00:00,2024-11-08 13:00:00+01:00,1007,North Sea,53.813705,5.722543,9.712363,106.755829,100,7.80,7.149994,1029.699951,1.279765,0.0
211241,2025-12-28 00:00:00+00:00,2025-12-28 22:00:00+00:00,2025-12-28 23:00:00+01:00,2002,Baltic Sea,54.797890,12.831684,13.388894,284.710938,100,5.10,4.450012,1018.799988,1.278533,0.0
175829,2024-07-15 00:00:00+00:00,2024-07-15 21:00:00+00:00,2024-07-15 23:00:00+02:00,2001,Baltic Sea,54.657291,12.248521,1.650757,54.865723,100,18.15,17.500000,1013.000000,1.214176,0.0
168255,2026-01-02 00:00:00+00:00,2026-01-03 14:00:00+00:00,2026-01-03 15:00:00+01:00,1008,North Sea,54.305798,5.273438,20.411087,316.885986,100,4.25,3.600006,998.299988,1.256655,0.1
7315,2024-12-20 00:00:00+00:00,2024-12-21 07:00:00+00:00,2024-12-21 08:00:00+01:00,1001,North Sea,54.868187,6.607143,13.146863,256.809387,100,8.75,8.100006,1005.400024,1.245342,0.0
266055,2024-12-12 00:00:00+00:00,2024-12-13 07:00:00+00:00,2024-12-13 08:00:00+01:00,2005,Baltic Sea,55.149384,14.220000,4.574112,195.859436,100,4.90,4.250000,1033.400024,1.297790,0.0
253128,2025-11-09 00:00:00+00:00,2025-11-10 01:00:00+00:00,2025-11-10 02:00:00+01:00,2004,Baltic Sea,55.008785,13.625499,3.233033,238.276443,100,8.85,8.200012,1014.500000,1.256168,0.0
116527,2025-02-08 00:00:00+00:00,2025-02-08 22:00:00+00:00,2025-02-08 23:00:00+01:00,1006,North Sea,54.094902,7.339806,8.676405,115.974342,100,4.90,4.250000,1028.300049,1.291385,0.0
130211,2024-04-11 00:00:00+00:00,2024-04-12 07:00:00+00:00,2024-04-12 09:00:00+02:00,1007,North Sea,53.813705,5.722543,12.543125,234.409134,100,10.65,10.000000,1027.000000,1.263561,0.0
201851,2025-01-01 00:00:00+00:00,2025-01-02 09:00:00+00:00,2025-01-02 10:00:00+01:00,2002,Baltic Sea,54.797890,12.831684,13.446190,300.379059,100,2.65,2.000000,1003.099976,1.270039,0.0


### Dropping columns unnecessary for the dataset

`region` – decided to keep it for now, as I will use it later for regional aggregation<br>
`latitude` and `longitude` – dropped; these are constant values ​​for each `point_id`, are not used in machine learning, and would only be needed for debugging<br>
`temperature_2m` – dropped; the dataframe already contains a hub-height-adjusted temperature, which is more useful<br>
`height_m` – dropped; a constant column that provides no information for the model<br>
`datetime_utc` – decided to keep it for now; it makes more sense to use local time as a cyclical feature, but I use this later for debugging<br>

In [5]:
# List of columns to drop from the final dataframe
COLS_TO_DROP = [
    # 'region',         # point_id has already the region information (1000 = North, 2000 = Baltic), so this column is redundant
    'latitude',         # constant latitude parameter, needed for debugging
    'longitude',        # constant longitude parameter, needed for debugging
    'temperature_2m',   # corrected temperature at hub height will be sufficient
    'height_m'          # constant column, can be removed
    # 'datetime_utc'    # optionally, local time would be more appropriate for the feature
    ]

# Columns dropping from the final dataframe
df_clean = df_complete.drop(COLS_TO_DROP, axis=1).copy()

# Displaying the columns of the cleaned dataframe
print(df_clean.columns)

Index(['datetime_run_utc', 'datetime_utc', 'datetime_berlin', 'point_id',
       'region', 'wind_speed', 'wind_direction', 'temperature_hub',
       'surface_pressure', 'air_density', 'precipitation'],
      dtype='str')


### Renaming the remaining columns for brevity and standardization

In [6]:
# Creating a dictionary to rename columns
rename_dict = {
    'datetime_run_utc': 'dt_run_utc',
    'datetime_utc':     'dt_utc',
    'datetime_berlin':  'dt_berlin',
    'wind_speed':       'w_speed',
    'wind_direction':   'w_dir',
    'temperature_hub':  'temp',
    'surface_pressure': 'press',
    'air_density':      'density',
}

# Renaming
df_clean = df_clean.rename(columns=rename_dict)

# Displaying the result
df_clean.sample(5)

,dt_run_utc,dt_utc,dt_berlin,point_id,region,w_speed,w_dir,temp,press,density,precipitation
113332,2024-10-09 00:00:00+00:00,2024-10-10 01:00:00+00:00,2024-10-10 03:00:00+02:00,1006,North Sea,4.978454,33.530495,14.300018,983.099976,1.191455,0.0
111722,2024-08-08 00:00:00+00:00,2024-08-09 03:00:00+00:00,2024-08-09 05:00:00+02:00,1006,North Sea,15.448705,209.468781,18.450012,1008.500000,1.204844,0.0
61831,2026-02-28 00:00:00+00:00,2026-03-01 03:00:00+00:00,2026-03-01 04:00:00+01:00,1003,North Sea,12.934160,259.081055,3.399994,1016.700012,1.280742,0.0
255136,2026-01-24 00:00:00+00:00,2026-01-25 07:00:00+00:00,2026-01-25 08:00:00+01:00,2004,Baltic Sea,11.189281,114.274429,-0.149994,1009.500000,1.288208,0.1
299511,2026-03-12 00:00:00+00:00,2026-03-13 02:00:00+00:00,2026-03-13 03:00:00+01:00,2006,Baltic Sea,14.059339,195.469635,4.500000,1012.200012,1.270022,0.0


### Saving the cleaned DataFrame to a Parquet file

In [7]:
# Determining the start and end dates of the cleaned dataframe
clean_df_start = df_clean['dt_utc'].min()
clean_df_end = df_clean['dt_utc'].max()

# Converting the dates to strings
start_date_str = pd.to_datetime(clean_df_start).strftime('%Y-%m-%d')
end_date_str = pd.to_datetime(clean_df_end).strftime('%Y-%m-%d')

# Displaying the start and end dates of the cleaned dataframe
print(f"Cleaned dataframe covers the period from {start_date_str} to {end_date_str}")

# # Uncomment if necessary============================================================
# # Saving the cleaned dataframe in parquet format
# df_clean.to_parquet(CLEAN_DATA_DIR / f'offshore_singleruns_only_{start_date_str}_to_{end_date_str}.parquet')

Cleaned dataframe covers the period from 2024-03-14 to 2026-06-21


If the desired merged DataFrame is already present in the `CLEAN_DATA_DIR` directory, you can start from the current step

In the subsequent cells, I will combine **weather data** with **generation data**

In [8]:
# Importing the cleaned offshore dataframe from parquet
clean_offshore_df = pd.read_parquet(CLEAN_DATA_DIR / 'offshore_singleruns_only_2024-03-14_to_2026-06-21.parquet')

# Displaying the coverage of the cleaned offshore dataframe
min_datetime_utc = clean_offshore_df['dt_utc'].min()
max_datetime_utc = clean_offshore_df['dt_utc'].max()

print(f'The DF starts at: {min_datetime_utc}')
print(f'The DF starts at: {max_datetime_utc}')


The DF starts at: 2024-03-14 22:00:00+00:00
The DF starts at: 2026-06-21 22:00:00+00:00


In [ ]:
# Importing the function to collect generation data from ENTSO-E
from utils.entsoe_utils import collect_entsoe_data

# Setting the start and end dates for ENTSO-E data collection
START = '2024-03-13'
END = '2026-06-12'

# Setting the ENTSO-E API key, either from environment variables or a placeholder
API = os.getenv("ENTSOE_API_KEY") or "YOUR_API_KEY_HERE"

# I need the generation data only at the moment
GEN_QUERRIES = {"gen_actual": "query_generation"}

# # Uncomment if necessary============================================================
# Data collection from ENTSO-E is commented out to avoid unnecessary API calls during development
# actual_generation_data = collect_entsoe_data(
#     country_code='DE_LU',
#     queries=GEN_QUERRIES,
#     api_key=API,
#     max_days=30,               # try to reduce if it gives a lot of missing values
#     start=START,
#     end=END,
#     retries=2,                 # 2-3 should be enough
#     retry_delay=2.0
# )

### Columns renaming

In this cell, I clean and standardize the column names returned by entsoe-py, since the default labels often contain special characters and inconsistent capitalization.

The naming convention used here is:

`_act` — actual values observed at the requested timestamp

`_fc` — forecast values for the same timestamp, produced in the past

**A critical detail**: although the forecast is requested with parameter `'A01'` (Day Ahead), ENTSO‑E does not guarantee that these values represent what traders actually saw before the gate closure. According to the **ENTSO‑E MoP documentation**, forecast data may be overwritten by more recent versions, which can introduce data leakage.

Energy load forecast `load_fc` for the next day must be published at least two hours before the Day Ahead **market gate closure** (at 12:00 local time), which is good.<br>

However, Day Ahead forecasts for total generation (`gen_fc`) or RES generation (`gen_fc_solar` for example) must be published before **18:00 on D‑1**. This means the values you download may reflect updated forecasts rather than the original Day Ahead view, making them unsafe to use directly as targets or features in Day Ahead training.

In [ ]:
# # IMPORTANT
# # I also tried applying this name-correction logic directly to the `collect_entsoe_data` function,
# # though not to all columns yet.
# # So, it might not be necessary to do this during the data cleaning stage.

# # removing special characters and converting column names to lowercase
# actual_generation_data.columns = [str(col).replace("('", "").replace("')", "").replace("', '", "_").replace(" ", "_").lower() for col in actual_generation_data.columns]

# Mapping for renaming columns
columns_rename_map = {
    # day ahead prices (auction)
    'prices_prices': 'prices',

    # Total energy load forecast for the country-zone
    'forecasted_load_load_forecast': 'load_fc',

    # Total energy generation forecast for the country-zone
    'gen_total_forecast_gen_total_forecast': 'gen_fc',

    # Total solar generation forecast for the country-zone
    'solar_gen_forecast_res': 'gen_fc_solar',

    # Total offshore wind generation forecast for the country-zone
    'wind_offshore_gen_forecast_res': 'gen_fc_wind_off',

    # Total onshore wind generation forecast for the country-zone
    'wind_onshore_gen_forecast_res': 'gen_fc_wind_on',

    # Actual total energy load for the country-zone
    'actual_load_load_actual': 'load_act',

    # Actual biomass generation
    'biomass_gen_actual_actual_aggregated_gen_actual': 'gen_biomass_act',

    # Actual generation from low-quality brown coal
    'fossil_brown_coal/lignite_gen_actual_actual_aggregated_gen_actual': 'gen_lignite_act',

    # Actual generation from coal-derived gas (small share)
    'fossil_coal-derived_gas_gen_actual_actual_aggregated_gen_actual': 'gen_coal_gas_act',

    # Actual generation from natural gas (NG) – gas power plants, CCGT, OCGT
    'fossil_gas_gen_actual_actual_aggregated_gen_actual': 'gen_gas_act',

    # Actual generation from hard coal
    'fossil_hard_coal_gen_actual_actual_aggregated_gen_actual': 'gen_hard_coal_act',

    # Actual generation from oil and diesel
    'fossil_oil_gen_actual_actual_aggregated_gen_actual': 'gen_oil_act',

    # Actual generation from geothermal plants
    'geothermal_gen_actual_actual_aggregated_gen_actual': 'gen_geothermal_act',

    # Actual generation from pumped storage hydro plants
    'hydro_pumped_storage_gen_actual_actual_aggregated_gen_actual': 'gen_hydro_pumped_act',

    # Actual consumption from pumped storage hydro plants (pumping mode)
    'hydro_pumped_storage_gen_actual_actual_consumption_gen_actual': 'gen_hydro_pumped_cons_act',

    # Actual generation from run-of-river and poundage hydro plants
    'hydro_run-of-river_and_poundage_gen_actual_actual_aggregated_gen_actual': 'gen_hydro_river_act',

    # Actual generation from water reservoirs, without pumps, works only on release
    'hydro_water_reservoir_gen_actual_actual_aggregated_gen_actual': 'hydro_res_act',

    # Actual generation from other non-renewable sources
    'other_gen_actual_actual_aggregated_gen_actual': 'gen_other_act',

    # Actual generation from other renewable sources
    'other_renewable_gen_actual_actual_aggregated_gen_actual': 'gen_other_ren_act',

    # Actual generation from solar panels
    'solar_gen_actual_actual_aggregated_gen_actual': 'gen_solar_act',

    # Actual generation from waste incineration
    'waste_gen_actual_actual_aggregated_gen_actual': 'gen_waste_act',

    # Actual generation from offshore wind turbines
    'wind_offshore_gen_actual_actual_aggregated_gen_actual': 'gen_wind_off_act',

    # Actual generation from onshore wind turbines
    'wind_onshore_gen_actual_actual_aggregated_gen_actual': 'gen_wind_on_act' 
}

# Uncomment if necessary============================================================
# # Renaming columns using the mapping defined above
# actual_generation_data = actual_generation_data.rename(columns=columns_rename_map)

# # Saving the cleaned generation data in parquet format
# actual_generation_data.to_parquet(RAW_DATA_DIR / 'gen_act_per_type_2024-03-13_to_2026-06-11.parquet')

### Loading stored offshore wind generation data, if available, and performing a basic check of the numerical distributions.

In [10]:
# Importing the cleaned generation data from parquet
actual_generation_data = pd.read_parquet(RAW_DATA_DIR / 'gen_act_per_type_2024-03-13_to_2026-06-11.parquet')

# Displaying descriptive statistics
actual_generation_data['gen_wind_off_act'].describe()

count    78816.000000
mean      2976.529868
std       2103.208779
min          0.000000
25%       1111.694750
50%       2618.002000
75%       4685.146750
max       8486.407000
Name: gen_wind_off_act, dtype: float64

### Downloading Installed Capacity Data

At this stage, I load data on installed capacity; this serves as the basis for calculating the **capacity factor** (the ratio of actual output to maximum potential output), which acts as the target variable for this machine learning task.

I chose this approach because I want the model to generalize more effectively across long historical sequences, accounting for the natural expansion of wind farms. The underlying concept is that a specific wind strength corresponds to a certain percentage of generation relative to total installed capacity, rather than a fixed value in megawatts. This also gives the model an inherent ability to **forecast system behavior** as new wind power plants come online.

In other words, I am intentionally training the model to operate within an evolving energy system rather than a static one.

In [11]:
# # Uncomment if necessary============================================================

# # Importing the function to get installed capacity for Wind Offshore
# from utils.get_installed_capacity import get_installed_power

# # Getting the installed capacity
# wind_offshore_capacity = get_installed_power(
#         start="2024-01-01",
#         end="2026-12-31",
#         sources=["Wind offshore"]
#     )

# # Renaming columns for clarity
# wind_offshore_capacity.rename(columns={'date': 'dt_utc', 'Wind offshore': 'offshore_capacity'}, inplace=True)

# # Defining the start and end dates for the installed capacity data
# wind_offshore_capacity_start_str = wind_offshore_capacity['dt_utc'].min().strftime('%Y-%m-%d')
# wind_offshore_capacity_end_str   = wind_offshore_capacity['dt_utc'].max().strftime('%Y-%m-%d')

# # Saving the installed capacity data in parquet format
# wind_offshore_capacity.to_parquet(RAW_DATA_DIR / f'wind_offshore_installed_capacity_{wind_offshore_capacity_start_str}_to_{wind_offshore_capacity_end_str}.parquet')

### Loading stored offshore wind installed capacity data, if available

These data are presented in gigawatts (GW)

In [12]:
# Reading the installed capacity data from parquet
wind_offshore_capacity = pd.read_parquet(RAW_DATA_DIR / 'wind_offshore_installed_capacity_2024-01-01_to_2026-07-01.parquet')

# Showing the first 5 rows
wind_offshore_capacity.head(10)

,dt_utc,offshore_capacity
0,2024-01-01 00:00:00+00:00,8.473
1,2024-02-01 00:00:00+00:00,8.473
2,2024-03-01 00:00:00+00:00,8.473
3,2024-04-01 00:00:00+00:00,8.473
4,2024-05-01 00:00:00+00:00,8.695
5,2024-06-01 00:00:00+00:00,8.851
6,2024-07-01 00:00:00+00:00,9.040
7,2024-08-01 00:00:00+00:00,9.127
8,2024-09-01 00:00:00+00:00,9.215
9,2024-10-01 00:00:00+00:00,9.215


### Preview of data from all three sources

Weather data are presented with a **1-hour** resolution in SI units, except for the direction (`w_dir`), which is in degrees rather than radians.

Installed capacity data have **a 1-month** resolution in Gigawatts (GW).

Generation data have a **15-minute** resolution in Megawatts (MW).

In [13]:
print('Weather data:\n',
      clean_offshore_df[['dt_utc', 'w_speed', 'w_dir', 'temp']].sample(5),
      '\n\n============================================\n\n')

print('Installed capacity wind offshore:\n',
      wind_offshore_capacity.sample(5),
      '\n\n============================================\n\n')

print('Actual offshore generation:\n',
      actual_generation_data[['gen_wind_off_act']].sample(5))

Weather data:
                           dt_utc    w_speed       w_dir       temp
151682 2024-04-08 03:00:00+00:00  11.997189  227.533600   9.250000
107746 2026-06-16 04:00:00+00:00   5.610035  306.665710  12.149994
34065  2025-07-07 04:00:00+00:00   7.460060  307.100403  14.050018
11029  2025-05-12 03:00:00+00:00  12.314727  123.238434  10.500000
282162 2024-05-16 21:00:00+00:00  15.686937  109.359085  12.550018 



Installed capacity wind offshore:
                       dt_utc  offshore_capacity
5  2024-06-01 00:00:00+00:00              8.851
4  2024-05-01 00:00:00+00:00              8.695
8  2024-09-01 00:00:00+00:00              9.215
19 2025-08-01 00:00:00+00:00              9.215
29 2026-06-01 00:00:00+00:00             10.601 



Actual offshore generation:
                            gen_wind_off_act
2025-08-18 05:45:00+00:00            55.146
2024-04-15 21:30:00+00:00          3378.280
2026-03-30 14:45:00+00:00          4819.849
2025-10-20 01:00:00+00:00          6178.253
202

### Feature engineering

From this point on, I began transforming fundamental physical quantities into features that were more "understandable" for the model.

In this cell, I created the following binary flags:

`cut_in_risk` – the wind farm might fail to reach the cut-in threshold, causing generation in the region to drop sharply or fall to zero.

`cut_out_risk` – the wind farm might trigger an emergency shutdown (protective mode) to avoid structural damage during storm conditions; similar to `cut_in_risk`, generation can drop drastically, potentially to zero.

`saturation_risk` – a binary flag indicating the potential to reach the power plateau (where maximum available power is being extracted); despite the `_risk` suffix, this is actually the desired operating mode for the turbine.

`icing_risk` – the risk of blade icing, which can negatively impact generation by degrading the blades aerodynamic performance.

In [14]:
# Creating a copy of the cleaned offshore dataframe for further processing
offshore_dataset = clean_offshore_df.copy()

# Setting thresholds for various risk conditions
CUTIN_THRESHOLD             = 3
CUTOUT_THRESHOLD            = 25  # For different seas and turbines, this can vary, but generally 25 m/s is a standard threshold for most modern blades
SATURATION_THRESHOLD        = 12
ICING_TEMP_UP_THRESHOLD     = 2   # Often taken as +2°C, as the blade freezes faster at speed
ICING_TEMP_DOWN_THRESHOLD   = -10 # Often taken as -10°C, as the blade freezes faster at speed

# Creating cut-in and cut-out risk indicators
offshore_dataset['cut_in_risk']   = offshore_dataset['w_speed'].apply(lambda x: 1 if x < CUTIN_THRESHOLD else 0)
offshore_dataset['cut_out_risk']  = offshore_dataset['w_speed'].apply(lambda x: 1 if x >= CUTOUT_THRESHOLD else 0)

# Creating saturation risk indicator
saturation_condition = (offshore_dataset['w_speed'] >= SATURATION_THRESHOLD) & (offshore_dataset['w_speed'] < CUTOUT_THRESHOLD)
offshore_dataset['saturation_risk'] = saturation_condition.astype(int)

# Creating icing risk indicator
icing_condition = (
    (offshore_dataset['precipitation'] > 0) & 
    (offshore_dataset['temp'] <= ICING_TEMP_UP_THRESHOLD) & 
    (offshore_dataset['temp'] >= ICING_TEMP_DOWN_THRESHOLD)
)
offshore_dataset['icing_risk'] = icing_condition.astype(int)

### Next, I create continuous features for the models, such as:

`power_proxy` – the product of wind speed cubed and air density; roughly speaking, this represents the **kinetic energy** of the wind that the turbine converts into **electrical energy**.

`w_dir_sin` and `w_dir_cos` – wind direction encoded using sine and cosine; this makes the variable continuous and allows the model to understand that **1 degree and 360 degrees are essentially the same thing**.

`doy_sin`, `doy_cos`, `hour_sin`, and `hour_cos` follow the same logic for creating cyclical time features.

I then empirically determined the aggregation methods for these parameters to optimize the number of features; as it turned out, the most important features were the power_proxy values ​​calculated individually for **each geographic location**.

Furthermore, I decided that **the final dataset would have a 15-minute resolution—matching** the resolution of the target variable.

I opted against interpolating the weather data, as I believe there may be complex, non-linear meteorological dependencies involved.

Instead, I added a feature indicating which 15-minute interval within the hour the observation falls into. The idea is to encourage the model to look more actively at the T-1 lag when processing the 0- or 15-minute marks, and at the T+1 lag for the 30- and 45-minute marks.

In [15]:
# Re-importing pandas and NumPy, just in case.
import pandas as pd
import numpy as np

# Ensuring that the time columns are in datetime format for further processing
time_cols = ['dt_run_utc', 'dt_utc', 'dt_berlin']
for col in time_cols:
    offshore_dataset[col] = pd.to_datetime(offshore_dataset[col])

# Creating a copy of the offshore dataset for further processing
df = offshore_dataset.copy()

# Column that identifies the measurement point (probe) in the offshore dataset. This is crucial for pivoting and aggregating data by point.
POINT_COL = 'point_id' 

# ---------------------------------------------------------
# Preparation of basic features before aggregation
# ---------------------------------------------------------
# Product of the cube of wind speed and air density, which serves as a proxy for power generation potential
df['power_proxy'] = (df['w_speed'] ** 3) * df['density']

# Converting wind direction from degrees to radians,
# and calculating its sine and cosine components for circular statistics
df['w_dir_rad'] = np.deg2rad(df['w_dir'])
df['w_dir_sin'] = np.sin(df['w_dir_rad'])
df['w_dir_cos'] = np.cos(df['w_dir_rad'])

# ---------------------------------------------------------
# Pivoting raw power_proxy values by 14 measurement points
# ---------------------------------------------------------
# Select only time, point, and proxy columns, then pivot the points into columns
proxy_wide = df.pivot_table(
    index=['dt_run_utc', 'dt_utc', 'dt_berlin'], 
    columns=POINT_COL, 
    values='power_proxy',
    aggfunc='first'
)

# Renaming the columns to have a clear and consistent naming convention
proxy_wide.columns = [f"point_{str(col).replace(' ', '_')}_power_proxy" for col in proxy_wide.columns]
proxy_wide = proxy_wide.reset_index()


# ---------------------------------------------------------
# Aggregation of other features by regions
# ---------------------------------------------------------
# Removed power_proxy from aggregation
agg_funcs = {
    # 'power_proxy': ['mean', 'std'], # Uncomment if you want to keep the mean and std of power_proxy by region, besides the point-wise values
    'w_dir_sin': ['mean'],
    'w_dir_cos': ['mean'],
    'temp': ['mean'],
    'press': ['std'], # For pressure, the standard deviation might be more informative than the mean
    'precipitation': ['mean'],
    'cut_in_risk': ['mean'],
    'cut_out_risk': ['mean'],
    'saturation_risk': ['mean'],
    'icing_risk': ['mean']
}

# Grouping by time and region, and applying the aggregation functions defined above
grouped = df.groupby(['dt_run_utc', 'dt_utc', 'dt_berlin', 'region']).agg(agg_funcs)

# Collapsing the multi-index columns into single-level columns for easier access and manipulation
grouped.columns = ['_'.join(col).strip() for col in grouped.columns.values]
grouped = grouped.reset_index()

# Adding a circular variance feature for wind direction, which is useful for understanding the variability of wind direction
grouped['w_dir_circ_var'] = 1 - np.sqrt(grouped['w_dir_sin_mean']**2 + grouped['w_dir_cos_mean']**2)

# Pivoting the grouped data to have regions as columns, which allows for easier comparison and analysis across different regions
region_pivot = grouped.pivot(index=['dt_run_utc', 'dt_utc', 'dt_berlin'], columns='region')
region_pivot.columns = [f"{col[1].replace(' ', '_')}_{col[0]}" for col in region_pivot.columns]
region_pivot = region_pivot.reset_index()


# ---------------------------------------------------------
# Merging points and regions
# ---------------------------------------------------------
pivot_df = pd.merge(proxy_wide, region_pivot, on=['dt_run_utc', 'dt_utc', 'dt_berlin'], how='inner')

# ---------------------------------------------------------
# Calculating lags (T-1, T+1) and filtering edges
# ---------------------------------------------------------
# Mandatory sorting by time for shift to work correctly
pivot_df = pivot_df.sort_values(['dt_run_utc', 'dt_utc'])

# Selecting feature columns (all except time columns)
feature_cols = [c for c in pivot_df.columns if c not in time_cols]

processed_runs = []
missing_lags_count = 0

for run_id, group in pivot_df.groupby('dt_run_utc'):
    # Difference: Previous hour minus Current
    diff_t_minus_1 = group[feature_cols].shift(1) - group[feature_cols]
    diff_t_minus_1.columns = [f"{c}_delta_before" for c in feature_cols]
    
    # Difference: Value in the next hour minus Current
    diff_t_plus_1 = group[feature_cols].shift(-1) - group[feature_cols]
    diff_t_plus_1.columns = [f"{c}_delta_after" for c in feature_cols]
    
    # Combine current values and deltas
    combined = pd.concat([group, diff_t_minus_1, diff_t_plus_1], axis=1)
    
    # Ensuring that both T-1 and T+1 are available for each row
    has_t_minus_1 = group['dt_utc'].shift(1).eq(group['dt_utc'] - pd.Timedelta(hours=1))
    has_t_plus_1  = group['dt_utc'].shift(-1).eq(group['dt_utc'] + pd.Timedelta(hours=1))

    # Keep only rows where both T-1 and T+1 are available
    valid_mask = has_t_minus_1 & has_t_plus_1

    # Counting the number of rows removed due to missing lags
    missing_lags_count += (~valid_mask).sum() - 2 # -2 because the first and last rows of each run will always be invalid due to lack of neighbors

    # Debugging output to show how many rows were removed for each run
    # print(f"Run {run_id}: удалено {(~valid_mask).sum()} наблюдений из-за отсутствия T-1 или T+1")

    # Appending the valid rows to the processed_runs list
    processed_runs.append(combined[valid_mask])

# Displaying the total number of rows with missing lags across all runs
print(f"Number of rows with missing lags inside each run: {missing_lags_count}")

# Concatenating all processed runs into a single DataFrame and resetting the index
hourly_dataset = pd.concat(processed_runs).reset_index(drop=True)

# ---------------------------------------------------------
# Multiplying the hourly dataset to create a 15-minute dataset
# ---------------------------------------------------------
steps_df = pd.DataFrame({'minute_step': [0, 15, 30, 45], 'merge_key': 1})
hourly_dataset['merge_key'] = 1

dataset_15m = pd.merge(hourly_dataset, steps_df, on='merge_key').drop('merge_key', axis=1)
dataset_15m = dataset_15m.sort_values(['dt_run_utc', 'dt_utc', 'minute_step']).reset_index(drop=True)

dataset_15m['dt_utc'] = dataset_15m['dt_utc'] + pd.to_timedelta(dataset_15m['minute_step'], unit='m')
dataset_15m['dt_berlin'] = dataset_15m['dt_berlin'] + pd.to_timedelta(dataset_15m['minute_step'], unit='m')


# ---------------------------------------------------------
# Creating the lead_time feature - time until the forecast moment in hours
# ---------------------------------------------------------
dataset_15m['dt_utc_hour'] = dataset_15m['dt_utc'].dt.floor('h')
dataset_15m['lead_time'] = (dataset_15m['dt_utc_hour'] - dataset_15m['dt_run_utc']).dt.total_seconds() / 3600.0
dataset_15m = dataset_15m.drop(columns=['dt_utc_hour'])


# ---------------------------------------------------------
# Cyclical features for time of day and day of year
# ---------------------------------------------------------
time_of_day_hours = dataset_15m['dt_berlin'].dt.hour + dataset_15m['dt_berlin'].dt.minute / 60.0
dataset_15m['hour_sin'] = np.sin(2 * np.pi * time_of_day_hours / 24)
dataset_15m['hour_cos'] = np.cos(2 * np.pi * time_of_day_hours / 24)

day_of_year = dataset_15m['dt_berlin'].dt.dayofyear
dataset_15m['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
dataset_15m['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)

# Cleaning column names for regions (this will not affect point columns)
dataset_15m.columns = dataset_15m.columns.str.replace(r'^Baltic_Sea_', 'baltic_', regex=True)
dataset_15m.columns = dataset_15m.columns.str.replace(r'^North_Sea_', 'north_', regex=True)


# Checking the result
print(f"\nSize of final dataset: {dataset_15m.shape}")

# Выведем список всех колонок с прокси, чтобы убедиться, что их 14
proxy_cols = [c for c in dataset_15m.columns if 'power_proxy' in c and 'delta' not in c]
print(f"\nColumns with proxy values ({len(proxy_cols)}):")
print(proxy_cols)

print("\nExample data:")
display(dataset_15m[['dt_run_utc', 'dt_utc', 'dt_berlin', 'minute_step'] + proxy_cols[:3]].head(8))

Number of rows with missing lags inside each run: 0

Size of final dataset: (79484, 111)

Columns with proxy values (14):
['point_1001_power_proxy', 'point_1002_power_proxy', 'point_1003_power_proxy', 'point_1004_power_proxy', 'point_1005_power_proxy', 'point_1006_power_proxy', 'point_1007_power_proxy', 'point_1008_power_proxy', 'point_2001_power_proxy', 'point_2002_power_proxy', 'point_2003_power_proxy', 'point_2004_power_proxy', 'point_2005_power_proxy', 'point_2006_power_proxy']

Example data:


,dt_run_utc,dt_utc,dt_berlin,minute_step,point_1001_power_proxy,point_1002_power_proxy,point_1003_power_proxy
0,2024-03-14 00:00:00+00:00,2024-03-14 23:00:00+00:00,2024-03-15 00:00:00+01:00,0,3239.308838,3344.652588,2453.244141
1,2024-03-14 00:00:00+00:00,2024-03-14 23:15:00+00:00,2024-03-15 00:15:00+01:00,15,3239.308838,3344.652588,2453.244141
2,2024-03-14 00:00:00+00:00,2024-03-14 23:30:00+00:00,2024-03-15 00:30:00+01:00,30,3239.308838,3344.652588,2453.244141
3,2024-03-14 00:00:00+00:00,2024-03-14 23:45:00+00:00,2024-03-15 00:45:00+01:00,45,3239.308838,3344.652588,2453.244141
4,2024-03-14 00:00:00+00:00,2024-03-15 00:00:00+00:00,2024-03-15 01:00:00+01:00,0,3373.035889,3385.712646,2416.891602
5,2024-03-14 00:00:00+00:00,2024-03-15 00:15:00+00:00,2024-03-15 01:15:00+01:00,15,3373.035889,3385.712646,2416.891602
6,2024-03-14 00:00:00+00:00,2024-03-15 00:30:00+00:00,2024-03-15 01:30:00+01:00,30,3373.035889,3385.712646,2416.891602
7,2024-03-14 00:00:00+00:00,2024-03-15 00:45:00+00:00,2024-03-15 01:45:00+01:00,45,3373.035889,3385.712646,2416.891602


In [17]:
mask_dup = dataset_15m.duplicated(subset=['dt_run_utc', 'dt_utc'])
dataset_15m[mask_dup]

,dt_run_utc,dt_utc,dt_berlin,point_1001_power_proxy,point_1002_power_proxy,point_1003_power_proxy,point_1004_power_proxy,point_1005_power_proxy,point_1006_power_proxy,point_1007_power_proxy,point_1008_power_proxy,point_2001_power_proxy,point_2002_power_proxy,point_2003_power_proxy,point_2004_power_proxy,point_2005_power_proxy,point_2006_power_proxy,baltic_w_dir_sin_mean,north_w_dir_sin_mean,baltic_w_dir_cos_mean,north_w_dir_cos_mean,baltic_temp_mean,north_temp_mean,baltic_press_std,north_press_std,baltic_precipitation_mean,north_precipitation_mean,baltic_cut_in_risk_mean,north_cut_in_risk_mean,baltic_cut_out_risk_mean,north_cut_out_risk_mean,baltic_saturation_risk_mean,north_saturation_risk_mean,baltic_icing_risk_mean,north_icing_risk_mean,baltic_w_dir_circ_var,north_w_dir_circ_var,point_1001_power_proxy_delta_before,point_1002_power_proxy_delta_before,point_1003_power_proxy_delta_before,point_1004_power_proxy_delta_before,point_1005_power_proxy_delta_before,point_1006_power_proxy_delta_before,point_1007_power_proxy_delta_before,point_1008_power_proxy_delta_before,point_2001_power_proxy_delta_before,point_2002_power_proxy_delta_before,point_2003_power_proxy_delta_before,point_2004_power_proxy_delta_before,point_2005_power_proxy_delta_before,point_2006_power_proxy_delta_before,baltic_w_dir_sin_mean_delta_before,north_w_dir_sin_mean_delta_before,baltic_w_dir_cos_mean_delta_before,north_w_dir_cos_mean_delta_before,baltic_temp_mean_delta_before,north_temp_mean_delta_before,baltic_press_std_delta_before,north_press_std_delta_before,baltic_precipitation_mean_delta_before,north_precipitation_mean_delta_before,baltic_cut_in_risk_mean_delta_before,north_cut_in_risk_mean_delta_before,baltic_cut_out_risk_mean_delta_before,north_cut_out_risk_mean_delta_before,baltic_saturation_risk_mean_delta_before,north_saturation_risk_mean_delta_before,baltic_icing_risk_mean_delta_before,north_icing_risk_mean_delta_before,baltic_w_dir_circ_var_delta_before,north_w_dir_circ_var_delta_before,point_1001_power_proxy_delta_after,point_1002_power_proxy_delta_after,point_1003_power_proxy_delta_after,point_1004_power_proxy_delta_after,point_1005_power_proxy_delta_after,point_1006_power_proxy_delta_after,point_1007_power_proxy_delta_after,point_1008_power_proxy_delta_after,point_2001_power_proxy_delta_after,point_2002_power_proxy_delta_after,point_2003_power_proxy_delta_after,point_2004_power_proxy_delta_after,point_2005_power_proxy_delta_after,point_2006_power_proxy_delta_after,baltic_w_dir_sin_mean_delta_after,north_w_dir_sin_mean_delta_after,baltic_w_dir_cos_mean_delta_after,north_w_dir_cos_mean_delta_after,baltic_temp_mean_delta_after,north_temp_mean_delta_after,baltic_press_std_delta_after,north_press_std_delta_after,baltic_precipitation_mean_delta_after,north_precipitation_mean_delta_after,baltic_cut_in_risk_mean_delta_after,north_cut_in_risk_mean_delta_after,baltic_cut_out_risk_mean_delta_after,north_cut_out_risk_mean_delta_after,baltic_saturation_risk_mean_delta_after,north_saturation_risk_mean_delta_after,baltic_icing_risk_mean_delta_after,north_icing_risk_mean_delta_after,baltic_w_dir_circ_var_delta_after,north_w_dir_circ_var_delta_after,minute_step,lead_time,hour_sin,hour_cos,doy_sin,doy_cos


### Creating the target for training

As I mentioned earlier, I am using the **Capacity Factor** as my target, calculated as follows:
`gen_wind_off_act` / `capacity`

In [77]:
# Removing timezone information from datetime columns to avoid issues during merging
dataset_15m['dt_utc'] = dataset_15m['dt_utc'].dt.tz_localize(None)
wind_offshore_capacity['dt_utc'] = wind_offshore_capacity['dt_utc'].dt.tz_localize(None)
actual_generation_data.index = actual_generation_data.index.to_series().dt.tz_localize(None)

# Convert all datetime fields to datetime64[ns] to guarantee type compatibility during merging
dataset_15m['dt_utc'] = dataset_15m['dt_utc'].astype('datetime64[ns]')
wind_offshore_capacity['dt_utc'] = wind_offshore_capacity['dt_utc'].astype('datetime64[ns]')
actual_generation_data.index = actual_generation_data.index.astype('datetime64[ns]')

# Attach offshore capacity using merge_asof (low‑frequency monthly data aligned backward to 15‑minute timestamps)
dataset_15m = pd.merge_asof(
    dataset_15m.sort_values('dt_utc'),
    wind_offshore_capacity.sort_values('dt_utc'),
    on='dt_utc',
    direction='backward'
)

# Rename capacity column for clarity
dataset_15m = dataset_15m.rename(columns={'offshore_capacity': 'capacity'})

# Join actual generation data using strict timestamp matching (dt_utc must be the index for join)
dataset_15m = dataset_15m.set_index('dt_utc')

dataset_15m = dataset_15m.join(
    actual_generation_data[['gen_wind_off_act']],
    how='left'
)

# Compute capacity factor (actual generation in MW divided by installed capacity in GW → convert GW to MW)
dataset_15m['capacity_factor'] = dataset_15m['gen_wind_off_act'] / (dataset_15m['capacity'] * 1000)  # capacity in GW, and gen_wind_off_act in MW, therefore multiply by 1000

# Drop rows with missing values to ensure a clean dataset
dataset_15m = dataset_15m.dropna(how='any')

# # If you need to return dt_utc to a column:
# dataset_15m = dataset_15m.reset_index()

# Extract date range for naming the output file
dataset_min_date = dataset_15m.index.min().strftime('%Y-%m-%d')
dataset_max_date = dataset_15m.index.max().strftime('%Y-%m-%d')

# # Uncomment if necessary============================================================
# # Saving the final dataset in parquet format
# dataset_15m.to_parquet(DATASETS_DIR / f'offshore_dataset_15m_raw_14_points_proxy_{dataset_min_date}_to_{dataset_max_date}.parquet')


### Output of numerical data distributions

In [81]:
dataset_15m.describe()

,point_1001_power_proxy,point_1002_power_proxy,point_1003_power_proxy,point_1004_power_proxy,point_1005_power_proxy,point_1006_power_proxy,point_1007_power_proxy,point_1008_power_proxy,point_2001_power_proxy,point_2002_power_proxy,point_2003_power_proxy,point_2004_power_proxy,point_2005_power_proxy,point_2006_power_proxy,baltic_w_dir_sin_mean,north_w_dir_sin_mean,baltic_w_dir_cos_mean,north_w_dir_cos_mean,baltic_temp_mean,north_temp_mean,baltic_press_std,north_press_std,baltic_precipitation_mean,north_precipitation_mean,baltic_cut_in_risk_mean,north_cut_in_risk_mean,baltic_cut_out_risk_mean,north_cut_out_risk_mean,baltic_saturation_risk_mean,north_saturation_risk_mean,baltic_icing_risk_mean,north_icing_risk_mean,baltic_w_dir_circ_var,north_w_dir_circ_var,point_1001_power_proxy_delta_before,point_1002_power_proxy_delta_before,point_1003_power_proxy_delta_before,point_1004_power_proxy_delta_before,point_1005_power_proxy_delta_before,point_1006_power_proxy_delta_before,point_1007_power_proxy_delta_before,point_1008_power_proxy_delta_before,point_2001_power_proxy_delta_before,point_2002_power_proxy_delta_before,point_2003_power_proxy_delta_before,point_2004_power_proxy_delta_before,point_2005_power_proxy_delta_before,point_2006_power_proxy_delta_before,baltic_w_dir_sin_mean_delta_before,north_w_dir_sin_mean_delta_before,baltic_w_dir_cos_mean_delta_before,north_w_dir_cos_mean_delta_before,baltic_temp_mean_delta_before,north_temp_mean_delta_before,baltic_press_std_delta_before,north_press_std_delta_before,baltic_precipitation_mean_delta_before,north_precipitation_mean_delta_before,baltic_cut_in_risk_mean_delta_before,north_cut_in_risk_mean_delta_before,baltic_cut_out_risk_mean_delta_before,north_cut_out_risk_mean_delta_before,baltic_saturation_risk_mean_delta_before,north_saturation_risk_mean_delta_before,baltic_icing_risk_mean_delta_before,north_icing_risk_mean_delta_before,baltic_w_dir_circ_var_delta_before,north_w_dir_circ_var_delta_before,point_1001_power_proxy_delta_after,point_1002_power_proxy_delta_after,point_1003_power_proxy_delta_after,point_1004_power_proxy_delta_after,point_1005_power_proxy_delta_after,point_1006_power_proxy_delta_after,point_1007_power_proxy_delta_after,point_1008_power_proxy_delta_after,point_2001_power_proxy_delta_after,point_2002_power_proxy_delta_after,point_2003_power_proxy_delta_after,point_2004_power_proxy_delta_after,point_2005_power_proxy_delta_after,point_2006_power_proxy_delta_after,baltic_w_dir_sin_mean_delta_after,north_w_dir_sin_mean_delta_after,baltic_w_dir_cos_mean_delta_after,north_w_dir_cos_mean_delta_after,baltic_temp_mean_delta_after,north_temp_mean_delta_after,baltic_press_std_delta_after,north_press_std_delta_after,baltic_precipitation_mean_delta_after,north_precipitation_mean_delta_after,baltic_cut_in_risk_mean_delta_after,north_cut_in_risk_mean_delta_after,baltic_cut_out_risk_mean_delta_after,north_cut_out_risk_mean_delta_after,baltic_saturation_risk_mean_delta_after,north_saturation_risk_mean_delta_after,baltic_icing_risk_mean_delta_after,north_icing_risk_mean_delta_after,baltic_w_dir_circ_var_delta_after,north_w_dir_circ_var_delta_after,minute_step,lead_time,hour_sin,hour_cos,doy_sin,doy_cos,capacity,gen_wind_off_act,capacity_factor
count,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000,78620.000000

### Final Dataset Description
At this stage, I have prepared a clean dataset suitable for model training.

I kept the columns `capacity` and `gen_wind_off_act` in the final table because they may be useful later during debugging and validation. However, it is important to keep in mind that **these fields are not part of the core feature set**.

For the standard version of the dataset, I use the configuration with 14 individual `power_proxy` signals **without aggregation**. This variant consistently performs better than the aggregated proxy, and additional feature engineering on top of it did not provide meaningful improvements.